<a href="https://colab.research.google.com/github/hyeonji-jang65/econ81230-project-jang/blob/main/code/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Features and preprocessing

Builds the common and extended feature sets for the analysis sample from `01_sample_construction`.

* Reference waves: work history at **t1** (last working wave before claiming), claiming conditions at **t0**.
* Dollar amounts in 2022 dollars (CPI-U). Income refers to the year before the interview; assets to the interview date.
* Missing values: RAND structural codes recoded (e.g. `.u` unmarried, `.z`/`.s` no pension → 0), then a missing value at the
  reference wave is filled with the respondent's own value **one wave earlier**. Remaining missing values are imputed later,
  inside each CV fold (median + missing indicator).
* Outputs: `features_prefill.parquet` (before the fill), `features.parquet` (after), `missing_summary.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pyreadstat

In [ ]:
import warnings
import numpy as np, pandas as pd, pyreadstat
warnings.filterwarnings('ignore', category=RuntimeWarning)   # nanmean of all-NaN rows

PATH   = '/content/drive/MyDrive/HRS/randhrs1992_2022v1.dta'
SAMPLE = '/content/drive/MyDrive/HRS/sample_ids.parquet'
OUTDIR = '/content/drive/MyDrive/HRS/'

samp = pd.read_parquet(SAMPLE)
samp = samp[samp['in_main']].reset_index(drop=True)
print('Main sample:', samp['type'].value_counts().to_dict())

W = range(1, 17)
FIXED = ['hhidpn', 'rabyear', 'ragender', 'raedyrs', 'raracem', 'rahispan', 'ravetrn', 'hacohort']
STEMS = {                        # prefix -> stems loaded for every wave
    'r': ['iwendy', 'agey_e', 'iearn', 'jhours', 'jcten', 'penct',
          'dcbal1', 'dcbal2', 'dcbal3', 'dcbal4',
          'jcocc', 'jcoccb', 'jcoccc', 'jcind', 'jcindb', 'jcindc',
          'mstat', 'isret', 'cendiv', 'shlt', 'liv75', 'cog27'],
    'h': ['atotb', 'atotf', 'aira', 'ahous', 'itot'],
    's': ['iearn', 'peninc'],
}
def vname(p, stem, w):
    return f'{p}{w}{stem}'

_, meta = pyreadstat.read_dta(PATH, metadataonly=True)
lower = {c.lower(): c for c in meta.column_names}
cols = [lower[v] for v in FIXED if v in lower]
cols += [lower[vname(p, s, w)] for p, ss in STEMS.items() for s in ss for w in W
         if vname(p, s, w) in lower]
missing_stems = [f'{p}*{s}' for p, ss in STEMS.items() for s in ss
                 if not any(vname(p, s, w) in lower for w in W)]
print('Stems not found in the RAND file:', missing_stems or 'none')

# user_missing=True keeps RAND special missing codes (.a .d .m .r .s .u .z ...)
raw, meta2 = pyreadstat.read_dta(PATH, usecols=cols, user_missing=True)
raw.columns = raw.columns.str.lower()
raw = samp[['hhidpn', 'type', 't0', 't1', 'ln_pen', 'pen_m']].merge(raw, on='hhidpn', how='left')
labels = {k.lower(): v for k, v in meta2.variable_value_labels.items()}
N = len(raw)
print(raw.shape)

In [ ]:
# CPI-U, all items, U.S. city average (FRED series CPIAUCSL, seasonally adjusted, 1982-84 = 100),
# annual average of the 12 monthly values. Source file: reference/CPIAUCSL.csv
CPI = {1990: 130.658, 1991: 136.167, 1992: 140.308, 1993: 144.475, 1994: 148.225,
       1995: 152.383, 1996: 156.858, 1997: 160.525, 1998: 163.008, 1999: 166.583,
       2000: 172.192, 2001: 177.042, 2002: 179.867, 2003: 184.000, 2004: 188.908,
       2005: 195.267, 2006: 201.558, 2007: 207.344, 2008: 215.254, 2009: 214.565,
       2010: 218.076, 2011: 224.923, 2012: 229.586, 2013: 232.952, 2014: 236.715,
       2015: 237.002, 2016: 240.005, 2017: 245.121, 2018: 251.100, 2019: 255.653,
       2020: 258.856, 2021: 270.973, 2022: 292.626, 2023: 304.703}
BASE_YEAR = 2022
_cpi = np.full(2100, np.nan)
for y, v in CPI.items():
    _cpi[y] = v

def to_real(year):
    """Multiplier converting nominal dollars of `year` into BASE_YEAR dollars."""
    year = np.asarray(year, float)
    out = np.full(year.shape, np.nan)
    ok = np.isfinite(year)
    out[ok] = CPI[BASE_YEAR] / _cpi[year[ok].astype(int)]
    return out

In [ ]:
idx = np.arange(N)
t0 = raw['t0'].to_numpy(int)
t1 = raw['t1'].to_numpy(int)

def split(col):
    """numeric values and RAND special-missing letter ('' if none) for one column."""
    s = raw[col]
    num = pd.to_numeric(s, errors='coerce').to_numpy(float)
    code = s.astype(str).str.strip().str.lower().str.lstrip('.').to_numpy(object)
    code = np.where(s.isna().to_numpy() | ~np.isnan(num), '', code)
    return num, code

_cache = {}
def M(p, stem):
    """(N, 17) numeric array and code array for prefix/stem; column w = wave w."""
    key = (p, stem)
    if key not in _cache:
        num = np.full((N, 17), np.nan)
        code = np.full((N, 17), '', dtype=object)
        for w in W:
            n = vname(p, stem, w)
            if n in raw:
                num[:, w], code[:, w] = split(n)
        _cache[key] = (num, code)
    return _cache[key]

def at(arr, t):
    out = np.full(N, np.nan) if arr.dtype != object else np.full(N, '', dtype=object)
    ok = (t >= 1) & (t <= 16)
    out[ok] = arr[idx[ok], t[ok]]
    return out

# interview year per wave (fallback: nominal wave year) and CPI multipliers
nominal_year = 1990 + 2 * np.arange(17)
iwy = M('r', 'iwendy')[0].copy()
iwy = np.where(np.isnan(iwy), nominal_year[None, :], iwy)
REAL_INCOME = to_real(iwy - 1)       # income amounts refer to the previous calendar year
REAL_STOCK  = to_real(iwy)           # assets and balances are valued at the interview

def get(p, stem, t, zero_codes=(), real=None, fill=True):
    """Value at wave t (before, after) filling from wave t-1.

    zero_codes : RAND codes that mean a structural zero (e.g. .u unmarried, .z no pension)
    real       : None, 'income' or 'stock' (CPI conversion to BASE_YEAR dollars)
    fill       : if the value at t is missing, use the respondent's own value at t-1
    """
    num, code = M(p, stem)
    num = num.copy()
    num[np.isin(code, list(zero_codes))] = 0
    if real == 'income':
        num = num * REAL_INCOME
    elif real == 'stock':
        num = num * REAL_STOCK
    before = at(num, t)
    after = before.copy()
    if fill:
        prev = at(num, t - 1)
        use = np.isnan(after) & ~np.isnan(prev)
        after[use] = prev[use]
    return before, after

def asinh_(x):
    return np.arcsinh(x)

def log_pos(x):
    """log for positive values, NaN otherwise."""
    x = np.asarray(x, float)
    return np.where(x > 0, np.log(np.where(x > 0, x, 1)), np.nan)

In [ ]:
# ── Occupation and industry: code-version fallback, then pooled groups ─────
# Following the reference code: use the first code version available at t1
# (jcocc -> jcoccb -> jcoccc; jcind -> jcindb -> jcindc). Each version uses a
# different census classification, so every version is first mapped to the same
# coarse groups from its own value labels and only then combined.
OCC_GROUPS = ['Managerial/professional', 'Sales/office', 'Service', 'Blue collar']
IND_GROUPS = ['Goods-producing', 'Trade/transport/utilities', 'Other private services',
              'Education/health/professional', 'Public administration']

OCC_RULES = [  # checked in order; first match wins
    (None,                      ['armed', 'military']),
    ('Managerial/professional', ['manag', 'exec', 'business', 'financ', 'prof', 'technic',
                                 'tech sup', 'computer', 'math', 'architect', 'engineer',
                                 'scien', 'community', 'social', 'legal', 'educ', 'librar',
                                 'arts', 'design', 'media', 'practitioner', 'diagnos']),
    ('Sales/office',            ['sales', 'clerical', 'office', 'admin']),
    ('Service',                 ['svc', 'serv', 'food', 'protect', 'personal', 'clean',
                                 'bldg', 'building', 'health', 'care']),
    ('Blue collar',             ['farm', 'forest', 'fish', 'constr', 'extract', 'install',
                                 'repair', 'mechanic', 'precision', 'product', 'operator',
                                 'transport', 'material', 'handler', 'labor', 'craft']),
]
IND_RULES = [
    (None,                            ['armed', 'military']),
    ('Public administration',         ['public ad', 'govt', 'government'], ['except']),
    ('Goods-producing',               ['agri', 'farm', 'forest', 'fish', 'mining', 'constr',
                                       'manuf', 'durable']),
    ('Trade/transport/utilities',     ['transp', 'util', 'commun', 'wholesale', 'retail',
                                       'trade', 'warehous']),
    ('Education/health/professional', ['prof', 'educ', 'health', 'social', 'hosp']),
    ('Other private services',        ['fin', 'insur', 'real est', 'bus', 'repair', 'personal',
                                       'entertain', 'recr', 'info', 'accom', 'food', 'svc',
                                       'serv', 'other']),
]

def classify(label, rules):
    s = str(label).lower()
    for rule in rules:
        group, keys, excl = (rule + ([],))[:3]
        if any(k in s for k in keys) and not any(e in s for e in excl):
            return group
    return 'UNMAPPED'

def label_of(lab, code):
    for k in (code, int(code), float(code)):
        if k in lab:
            return lab[k]
    return '?'

def group_version(stem, rules, t):
    """Group label at wave t for one code version (NaN if missing or excluded)."""
    num, _ = M('r', stem)
    out = np.full(N, None, dtype=object)
    v = at(num, t)
    mapping_rows = []
    for w in W:
        lab = labels.get(vname('r', stem, w), {})
        sel = (t == w) & ~np.isnan(v)
        for code in np.unique(v[sel]):
            text = label_of(lab, code)
            g = classify(text, rules)
            out[sel & (v == code)] = g
            mapping_rows.append((stem, w, int(code), text, g))
    return out, mapping_rows

def coalesce_groups(stems, rules, t):
    res = np.full(N, None, dtype=object)
    rows = []
    for st in stems:
        g, r = group_version(st, rules, t)
        rows += r
        take = (res == None) & (g != None)        # noqa: E711
        res[take] = g[take]
    res[np.isin(res, ['UNMAPPED'])] = None
    return res, pd.DataFrame(rows, columns=['version', 'wave', 'code', 'label', 'group'])

def grouped(stems, rules, t):
    """(before, after): t1 value, then filled from t1-1."""
    before, mp = coalesce_groups(stems, rules, t)
    prev, _ = coalesce_groups(stems, rules, t - 1)
    after = before.copy()
    use = (after == None) & (prev != None)        # noqa: E711
    after[use] = prev[use]
    return before, after, mp

occ_b, occ_a, occ_map = grouped(['jcocc', 'jcoccb', 'jcoccc'], OCC_RULES, t1)
ind_b, ind_a, ind_map = grouped(['jcind', 'jcindb', 'jcindc'], IND_RULES, t1)

# Check these tables: every label must sit in a sensible group
for name, mp in [('Occupation', occ_map), ('Industry', ind_map)]:
    tab = (mp.drop_duplicates(['version', 'label', 'group'])
             .sort_values(['version', 'group', 'code'])[['version', 'code', 'label', 'group']])
    print(f'\n{name} code -> group mapping (check UNMAPPED / None rows):')
    print(tab.to_string(index=False))

In [ ]:
# ── Build features: value at the reference wave (before) and after the t-1 fill ──
B, A_ = {}, {}                     # before / after fill
SET = {}                           # 'common' or 'extended'
LEVEL = {}                         # level variables used for descriptive tables

def add(name, pair, set_='common'):
    B[name], A_[name] = pair
    SET[name] = set_

# Fixed characteristics (no fill needed)
def fixed(col):
    v = pd.to_numeric(raw[col], errors='coerce').to_numpy(float) if col in raw else np.full(N, np.nan)
    return v, v.copy()
female = fixed('ragender'); add('female', tuple(np.where(np.isnan(x), np.nan, x == 2) for x in female))
add('educ_yrs', fixed('raedyrs'))
add('veteran', fixed('ravetrn'))
race, hisp = fixed('raracem')[0], fixed('rahispan')[0]
race_g = np.select([hisp == 1, race == 1, race == 2, race == 3],
                   ['Hispanic', 'White', 'Black', 'Other'], default=None).astype(object)
race_g[np.isnan(race) & np.isnan(hisp)] = None
add('race', (race_g, race_g.copy()))
add('birth_year', fixed('rabyear'))

# ── Work history at t1 ──
earn_t1 = get('r', 'iearn', t1, real='income')
earn_p1 = get('r', 'iearn', t1 - 1, real='income')
add('earn_t1', earn_t1)                                 # level, real annual
# final average salary: mean real earnings over t1 and t1-1 (available waves)
fas = tuple(np.nanmean(np.column_stack([a, b]), axis=1) for a, b in zip(earn_t1, earn_p1))
add('fas', fas)
add('tenure', get('r', 'jcten', t1))
add('hours', get('r', 'jhours', t1, zero_codes=('w',)))
add('occ', (occ_b, occ_a))
add('ind', (ind_b, ind_a))
add('n_plans', get('r', 'penct', t1, zero_codes=('z',)))
# DC balance: sum over plans at t1 (DC only); fill whole sum from t1-1 if missing
bal = [get('r', f'dcbal{k}', t1, real='stock') for k in range(1, 5)]
bal_prev = [get('r', f'dcbal{k}', t1 - 1, real='stock', fill=False)[0] for k in range(1, 5)]
def sum_plans(mats):
    m = np.column_stack(mats)
    s = np.nansum(m, axis=1); s[np.isnan(m[:, 0])] = np.nan
    return s
dc_b = sum_plans([b[0] for b in bal]); dc_prev = sum_plans(bal_prev)
dc_a = np.where(np.isnan(dc_b), dc_prev, dc_b)
is_dc = (raw['type'] == 'DC').to_numpy()
add('dc_balance', (np.where(is_dc, dc_b, np.nan), np.where(is_dc, dc_a, np.nan)))

# ── Claiming conditions at t0 ──
age_t0 = get('r', 'agey_e', t0)
add('claim_age', age_t0)
def age_group(a):
    g = np.select([a < 55, a < 60, a < 62, a < 65, a >= 65],
                  ['<55', '55-59', '60-61', '62-64', '65+'], default=None).astype(object)
    g[np.isnan(a)] = None
    return g
add('claim_age_grp', tuple(age_group(a) for a in age_t0))
def mstat_group(x):
    g = np.select([np.isin(x, [1, 2, 3]), np.isin(x, [4, 5, 6]), x == 7, x == 8],
                  ['Married/partnered', 'Separated/divorced', 'Widowed', 'Never married'],
                  default=None).astype(object)
    g[np.isnan(x)] = None
    return g
add('marital', tuple(mstat_group(x) for x in get('r', 'mstat', t0)))
ss = get('r', 'isret', t0, real='income')
add('ss_receipt', tuple(np.where(np.isnan(x), np.nan, x > 0) for x in ss))
def region(x):
    g = np.select([np.isin(x, [1, 2]), np.isin(x, [3, 4]), np.isin(x, [5, 6, 7]), np.isin(x, [8, 9])],
                  ['Northeast', 'Midwest', 'South', 'West'], default=None).astype(object)
    g[np.isnan(x)] = None
    return g
add('region', tuple(region(x) for x in get('r', 'cendiv', t0)))
gap = (t0 - t1).astype(float); add('gap_waves', (gap, gap.copy()))
cy = at(iwy, t0); add('claim_year', (cy, cy.copy()))

# ── Extended set (t1 values unless noted) ──
for stem, name in [('atotb', 'total_wealth'), ('atotf', 'fin_wealth'), ('aira', 'ira'),
                   ('ahous', 'home_value')]:
    add(name, get('h', stem, t1, real='stock'), 'extended')
add('hh_income', get('h', 'itot', t1, real='income'), 'extended')
add('sp_earn', get('s', 'iearn', t1, zero_codes=('u',), real='income'), 'extended')
sp_pen = get('s', 'peninc', t0, zero_codes=('u', 'z', 's'))
add('sp_pension', sp_pen, 'extended')
add('health', get('r', 'shlt', t1), 'extended')         # 1 excellent ... 5 poor
add('p_live75', get('r', 'liv75', t1), 'extended')
add('cognition', get('r', 'cog27', t1), 'extended')     # mode-adjusted, waves 3-15

print('Built', len(B), 'variables:', list(B))

In [ ]:
# ── Model features (transformations) and output ────────────────────────────
CAT = ['race', 'occ', 'ind', 'claim_age_grp', 'marital', 'region']
TRANSFORM = {                       # name -> (new column, function)
    'earn_t1':    ('ln_earn_t1', log_pos),
    'fas':        ('ln_fas', log_pos),
    'tenure':     ('ln_tenure', log_pos),
    'dc_balance': ('ln_dc_balance', log_pos),
    'total_wealth': ('asinh_total_wealth', asinh_),
    'fin_wealth': ('asinh_fin_wealth', asinh_),
    'ira':        ('asinh_ira', asinh_),
    'home_value': ('asinh_home_value', asinh_),
    'hh_income':  ('asinh_hh_income', asinh_),
    'sp_earn':    ('asinh_sp_earn', asinh_),
}

def frame(D):
    f = raw[['hhidpn', 'type', 't0', 't1', 'ln_pen', 'pen_m']].copy()
    for k, v in D.items():
        f[k] = v if k in CAT else pd.to_numeric(pd.Series(v), errors='coerce').to_numpy(float)
    for k, (new, fn) in TRANSFORM.items():
        f[new] = fn(f[k].to_numpy(float))
    return f

before, after = frame(B), frame(A_)

# Missing rates, by plan type: at the reference wave and after the t-1 fill
rows = []
for k in B:
    for tp in ['DB', 'DC']:
        if k == 'dc_balance' and tp == 'DB':        # not applicable
            continue
        s = before['type'] == tp
        mb = before.loc[s, k].isna().mean() * 100
        ma = after.loc[s, k].isna().mean() * 100
        rows.append((k, SET[k], tp, round(mb, 1), round(ma, 1), round(mb - ma, 1)))
miss = pd.DataFrame(rows, columns=['variable', 'set', 'type', 'miss_before_%', 'miss_after_%',
                                   'filled_from_t-1_%'])
print(miss.pivot_table(index=['set', 'variable'], columns='type',
                       values=['miss_before_%', 'miss_after_%']).round(1).to_string())

# Earnings at t1 should be positive for workers: report zero/negative values (set to NaN in logs)
print('\nNon-positive earnings at t1 (after fill):', (after['earn_t1'] <= 0).sum())

before.to_parquet(OUTDIR + 'features_prefill.parquet', index=False)
after.to_parquet(OUTDIR + 'features.parquet', index=False)
miss.to_csv(OUTDIR + 'missing_summary.csv', index=False)
pd.Series(SET).rename('set').to_csv(OUTDIR + 'feature_sets.csv')
print('\nSaved features_prefill.parquet, features.parquet, missing_summary.csv to', OUTDIR)
# Median imputation + missing indicators are applied later, inside each CV fold.